In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

### 1. Data Preprocessing

In [3]:
# 앵커 유저 데이터 로드
file_path = "../dataset/anchor_labeled_data.csv"
df = pd.read_csv(file_path)

# 비재무(Source) 범주형 컬럼 리스트 정의
categorical_cols = [
    'OCCAT1', 'OCCAT2', 'INDCAT', 'LF', 'HOUSECL', 'EDCL', 'EDUC', 
    'AGECL', 'LIFECL', 'FAMSTRUCT', 'KIDS', 'MARRIED', 'EXPENSHILO', 'WSAVED', 
    'SAVRES1', 'SAVRES2', 'SAVRES3', 'SAVRES4', 'SAVRES5', 
    'SAVRES6', 'SAVRES7', 'SAVRES8', 'SAVRES9'
]

# FIN(총 금융자산)이 0 초과인 유저만 필터링
df_filtered = df[df['FIN'] > 0].copy()

# 결측치가 있는 범주형 컬럼 확인
missing_counts = df_filtered[categorical_cols].isna().sum()
missing_cols = missing_counts[missing_counts > 0]
if len(missing_cols) > 0:
    print("결측값이 있는 범주형 컬럼:")
    print(missing_cols)
else:
    print("결측값이 있는 범주형 컬럼 없음")

# 제거할 행의 CASEID 추출
rows_before = len(df_filtered)
if 'CASEID' in df_filtered.columns:
    dropped_case_ids = df_filtered[df_filtered[categorical_cols].isna().any(axis=1)]['CASEID'].tolist()
    if dropped_case_ids:
        print(f"삭제된 행의 CASEID ({len(dropped_case_ids)}개): {dropped_case_ids}")
    else:
        print("삭제된 행의 CASEID 없음")
else:
    dropped_case_ids = []
    print("CASEID 컬럼이 없어 삭제된 행 ID를 출력할 수 없습니다.")

# 실제 제거 수행
rows_before = len(df_filtered)
df_filtered.dropna(subset=categorical_cols, inplace=True)
rows_after = len(df_filtered)
print(f"삭제된 행 수: {rows_before - rows_after}개 (범주형 결측값 포함)")

# 1-1. Source Domain (비재무 데이터) 전처리

# 모든 카테고리 변수를 float에서 int형으로 안전하게 변환 (PyTorch Embedding은 정수형을 받음)
for col in categorical_cols:
    df_filtered[col] = df_filtered[col].astype(int)

# 각 컬럼별 '최대값 + 1'을 구해서 임베딩 클래스 개수(num_classes)를 동적 계산
# (예: 값이 1,2,3,4 이면 PyTorch는 0~4 인덱스를 커버해야 하므로 5칸의 자리가 필요함)
categorical_cardinalities = [df_filtered[col].max() + 1 for col in categorical_cols]

# 모델 입력용 PyTorch 텐서로 변환
x_cat_tensor = torch.tensor(df_filtered[categorical_cols].values, dtype=torch.long)

# 1-2. Target Domain (재무 데이터) 전처리
# 자산군 그룹화 및 비율 계산
df_filtered['RATIO_CASH'] = (df_filtered['LIQ'] + df_filtered['CDS']) / df_filtered['FIN']
df_filtered['RATIO_RISK'] = df_filtered['EQUITY'] / df_filtered['FIN']
df_filtered['RATIO_BOND'] = df_filtered['BOND'] / df_filtered['FIN']
df_filtered['RATIO_PENSION'] = df_filtered['RETQLIQ'] / df_filtered['FIN']

# 4차원 비율 벡터만 추출 후 NaN 처리
ratio_columns = ['RATIO_CASH', 'RATIO_RISK', 'RATIO_BOND', 'RATIO_PENSION']
df_ratios = df_filtered[ratio_columns].fillna(0)

# 소수점 오차 방지를 위해 행(User) 단위로 다시 한 번 비율 L1 정규화 (합을 무조건 1로!)
df_ratios = df_ratios.div(df_ratios.sum(axis=1), axis=0).fillna(0)

# 모델 입력용 PyTorch 텐서로 변환
x_ratio_tensor = torch.tensor(df_ratios.values, dtype=torch.float32)

print(f"\n전처리 완료: 총 {len(df_filtered)}명의 유저 데이터 준비됨.")
print(f" - 비재무 텐서 크기: {x_cat_tensor.shape} (유저수, 23개 범주형 컬럼)")
print(f" - 재무 텐서 크기: {x_ratio_tensor.shape} (유저수, 4개 자산군 비율)\n")


결측값이 있는 범주형 컬럼:
EDUC    1
dtype: int64
삭제된 행의 CASEID (1개): [200306]
삭제된 행 수: 1개 (범주형 결측값 포함)

전처리 완료: 총 14676명의 유저 데이터 준비됨.
 - 비재무 텐서 크기: torch.Size([14676, 23]) (유저수, 23개 범주형 컬럼)
 - 재무 텐서 크기: torch.Size([14676, 4]) (유저수, 4개 자산군 비율)



### 2. Model Definition

In [4]:
from models import SourceEncoder, TargetEncoder

output_latent_dim = 128 # 두 벡터가 만나는 중간 공간의 차원

### 3. Forward Pass Test

In [5]:
# 모델 초기화
source_encoder = SourceEncoder(categorical_cardinalities, embed_dim=16, output_dim=output_latent_dim)
target_encoder = TargetEncoder(input_dim=4, output_dim=output_latent_dim)

# 전체 데이터 통과시켜보기 (메모리 절약을 위해 실제 학습 시에는 DataLoader로 Mini-batch 처리 필요)
source_encoder.eval()
target_encoder.eval()

with torch.no_grad():
    z_NF = source_encoder(x_cat_tensor)
    z_F = target_encoder(x_ratio_tensor)

print("모델 통과 결과:")
print(f" - 추출된 비재무 잠재 벡터 (z_NF): {z_NF.shape}")
print(f" - 추출된 재무 잠재 벡터 (z_F): {z_F.shape}")

모델 통과 결과:
 - 추출된 비재무 잠재 벡터 (z_NF): torch.Size([14676, 128])
 - 추출된 재무 잠재 벡터 (z_F): torch.Size([14676, 128])


### 4. Save Tensor

In [6]:
torch.save(x_cat_tensor, '../dataset/processed/x_cat_tensor.pt')
torch.save(x_ratio_tensor, '../dataset/processed/x_ratio_tensor.pt')
torch.save(torch.tensor(categorical_cardinalities), '../dataset/processed/cardinalities.pt')
print("데이터 저장 완료")

데이터 저장 완료


In [7]:
# RISK_LABEL 추출
labels = df['RISK_LABEL'].values

# 0부터 시작
if labels.min() == 1:
    labels = labels - 1
    print("라벨을 1~5에서 0~4로 조정했습니다.")

# 텐서로 변환 및 저장
labels_tensor = torch.tensor(labels, dtype=torch.long)
torch.save(labels_tensor, '../dataset/processed/labels_tensor.pt')

print("라벨 텐서 생성 완료:", labels_tensor.shape)

라벨을 1~5에서 0~4로 조정했습니다.
라벨 텐서 생성 완료: torch.Size([14677])
